## Werkzeuge

In [1]:
from pandas import read_csv
import pandas as pd
from matplotlib import pyplot as plt

from sklearn.cluster import KMeans

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Daten laden

In [8]:
data = read_csv(
    "../data/online_retail_data.csv",
    sep=";",
    skiprows=1,            # Erste Zeile überspringen
    decimal=",",           # Komma
    encoding="utf-8-sig", 
    on_bad_lines="skip",   # ERROR-Zeile überspringen
)
print(data.shape)
data.head()

(541910, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,01.12.10 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6.0,01.12.10 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,01.12.10 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,01.12.10 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,01.12.10 08:26,3.39,17850.0,United Kingdom


## Daten explorieren

In [9]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541908 non-null  str    
 1   StockCode    541676 non-null  str    
 2   Description  540018 non-null  str    
 3   Quantity     541908 non-null  float64
 4   InvoiceDate  541908 non-null  str    
 5   UnitPrice    541908 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541908 non-null  str    
dtypes: float64(3), str(5)
memory usage: 33.1 MB


In [12]:
data.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541908.000000,541908,541908.000000,406829.000000
mean,9.552262,2011-07-04 13:34:31.988861,4.611121,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081359,NaN,96.759942,1713.600303


## Daten vorbereiten

Daten bereinigen

In [ ]:
data = data.dropna(how="all")
data = data[data["InvoiceNo"].notna()]
data = data[~data["Country"].astype(str).str.strip().str.isdigit()]   # Country="0" rauslöschen
data = data[~data["InvoiceNo"].astype(str).str.startswith("C")]       # Storierte Rechungen rauslöschen
data = data[(data["Quantity"] > 0) & (data["UnitPrice"] > 0)]
data = data[data["CustomerID"].notna() & data["InvoiceDate"].notna()]
data["CustomerID"] = data["CustomerID"].astype(int)
data["TotalPrice"] = data["Quantity"] * data["UnitPrice"]

print(data.shape)
print("Eindeutige Kunden:", data["CustomerID"].nunique())

(397883, 9)
Eindeutige Kunden: 4338


Umsatz berechnen

In [15]:
data["LinePrice"] = data["Quantity"] * data["UnitPrice"]
data[["Quantity", "UnitPrice", "LinePrice"]].head()

,Quantity,UnitPrice,LinePrice
0,6.0,2.55,15.30
1,6.0,3.39,20.34
2,8.0,2.75,22.00
3,6.0,3.39,20.34
4,6.0,3.39,20.34


Umsatz pro Kunde und Revenue Zeile hinzufügen

In [18]:
revenue_df = data.groupby("CustomerID")["LinePrice"].sum().reset_index()
revenue_df = revenue_df.rename(columns={"LinePrice": "Revenue"})
print(revenue_df.shape)
revenue_df.head()

(4338, 2)


,CustomerID,Revenue
0,12346,77183.60
1,12347,4310.00
2,12348,1797.24
3,12349,1757.55
4,12350,334.40


Eine Zeille pro Kunde

In [19]:

kunden = data.groupby("CustomerID").agg(
    Revenue       =("LinePrice", "sum"),       # Gesamtumsatz je Kunde (= Zielwert)
    AnzahlArtikel =("StockCode", "nunique"),   # Anzahl verschiedener Produkte
    AnzahlKaeufe  =("InvoiceNo", "nunique"),   # Anzahl Bestellungen
    DurchschnPreis=("UnitPrice", "mean")       # mittlerer Artikelpreis
).reset_index()

print(kunden.shape)
kunden.head()

(4338, 5)


,CustomerID,Revenue,AnzahlArtikel,AnzahlKaeufe,DurchschnPreis
0,12346,77183.60,1,1,1.040000
1,12347,4310.00,103,7,2.644011
2,12348,1797.24,22,4,5.764839
3,12349,1757.55,73,1,8.289041
4,12350,334.40,17,1,3.841176


Daten vertikal Splitten

In [21]:
X = kunden[[
    "AnzahlArtikel",
    "AnzahlKaeufe",
    "DurchschnPreis"
]]

y = kunden["Revenue"]

Horizontal splitten

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(3470, 3) (3470,)
(868, 3) (868,)


Vorbereitungspipeline

In [23]:
numerical_cols = make_column_selector(dtype_include=["number"])
categorical_cols = make_column_selector(dtype_exclude=["number"])

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)
print(X_train.shape, X_test.shape)

(3470, 3) (868, 3)


## Maschine Learning